In [1]:
import numpy as np
import pandas as pd
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi':120,'font.size':10,
                     'axes.spines.top':False,'axes.spines.right':False})
SEED = 42
np.random.seed(SEED)
print('설정 완료')

설정 완료


In [2]:
BASE = r'c:\Users\kevin\OneDrive\Desktop\AISO\UNSW-NB15'

print('로딩 중...')
tr_raw = pd.read_csv(f'{BASE}/UNSW_NB15_training-set.csv')
te_raw = pd.read_csv(f'{BASE}/UNSW_NB15_testing-set.csv')
print(f'Train: {len(tr_raw):,}행 | Test: {len(te_raw):,}행 | 피처: {len(tr_raw.columns)}열')

print('\n=== Train 공격 유형 분포 ===')
print(tr_raw['attack_cat'].value_counts().to_string())
print('\n=== Test 공격 유형 분포 ===')
print(te_raw['attack_cat'].value_counts().to_string())

로딩 중...
Train: 82,332행 | Test: 175,341행 | 피처: 45열

=== Train 공격 유형 분포 ===
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44

=== Test 공격 유형 분포 ===
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130


In [3]:
# ── 전처리 ──────────────────────────────────────────────────
DROP_COLS = ['id', 'attack_cat', 'label']
CAT_COLS  = ['proto', 'service', 'state']

def preprocess(df, encoders=None, fit=False):
    d = df.copy()
    # 범주형: Label Encoding (proto 131종 → OHE 불가)
    if fit:
        encoders = {c: LabelEncoder().fit(d[c].astype(str)) for c in CAT_COLS}
    for c in CAT_COLS:
        d[c] = encoders[c].transform(
            d[c].astype(str).map(
                lambda x, enc=encoders[c]: x if x in enc.classes_ else enc.classes_[0]
            )
        )
    feat_cols = [c for c in d.columns if c not in DROP_COLS]
    return d[feat_cols].values.astype(float), encoders

# 이상 레이블 정리
tr_raw['attack_cat'] = tr_raw['attack_cat'].str.strip()
te_raw['attack_cat'] = te_raw['attack_cat'].str.strip()

# 희귀 공격 정의 (train 기준 < 500개)
tr_counts   = tr_raw['attack_cat'].value_counts()
rare_types  = set(tr_counts[(tr_counts < 500) &
                              (tr_counts.index != 'Normal')].index)
print('희귀 공격 (train holdout):',
      {t: int(tr_counts[t]) for t in rare_types})

# Train 풀에서 희귀 공격 제외 (학습에 절대 포함 안 함)
tr_common = tr_raw[~tr_raw['attack_cat'].isin(rare_types)].copy()

# 피처 행렬
X_tr_raw, encoders = preprocess(tr_common, fit=True)
y_tr_full = tr_common['label'].values
atk_tr    = tr_common['attack_cat'].values

# 불균형 서브샘플 (20K 정상 + 2K 이상)
rng = np.random.RandomState(SEED)
norm_idx = np.where(y_tr_full == 0)[0]
anom_idx = np.where(y_tr_full == 1)[0]

N_NORMAL = min(20000, len(norm_idx))
N_SEEN   = min(2000,  len(anom_idx))
sel_n = rng.choice(norm_idx, N_NORMAL, replace=False)
sel_a = rng.choice(anom_idx, N_SEEN,   replace=False)

X_imbal_raw = np.vstack([X_tr_raw[sel_n], X_tr_raw[sel_a]])
y_imbal     = np.array([0]*N_NORMAL + [1]*N_SEEN)
atk_imbal_a = pd.Series(atk_tr[sel_a])

# 스케일
scaler  = StandardScaler()
X_imbal = scaler.fit_transform(X_imbal_raw)

# Test set (공식 분리 그대로 사용)
X_te_raw, _ = preprocess(te_raw, encoders=encoders, fit=False)
X_test  = scaler.transform(X_te_raw)
y_test  = te_raw['label'].values
atk_test = te_raw['attack_cat'].values

# PCA for 최적화 샘플러 (20D)
pca = PCA(n_components=20, random_state=SEED)
X_imbal_pca = pca.fit_transform(X_imbal)
X_anom_pca  = X_imbal_pca[y_imbal == 1]

print(f'\n학습: {N_NORMAL:,} normal + {N_SEEN:,} anomaly'
      f'  (이상 {N_SEEN/(N_NORMAL+N_SEEN)*100:.1f}%)')
print(f'테스트: {len(X_test):,}')
print(f'\n학습 이상 유형:')
print(atk_imbal_a.value_counts().to_string())
print(f'\n테스트 희귀 공격 수:')
for t in sorted(rare_types):
    print(f'  {t}: {(atk_test==t).sum()}')

희귀 공격 (train holdout): {'Worms': 44, 'Shellcode': 378}

학습: 20,000 normal + 2,000 anomaly  (이상 9.1%)
테스트: 175,341

학습 이상 유형:
Generic           827
Exploits          504
Fuzzers           295
DoS               187
Reconnaissance    137
Analysis           28
Backdoor           22

테스트 희귀 공격 수:
  Shellcode: 1133
  Worms: 130


In [4]:
# ── 평가 함수 ────────────────────────────────────────────────
preds   = {}
results = {}

def evaluate(X_tr, y_tr, label=''):
    clf = GradientBoostingClassifier(n_estimators=150, random_state=SEED)
    clf.fit(X_tr, y_tr)
    prob = clf.predict_proba(X_test)[:,1]
    pred = (prob >= 0.5).astype(int)
    if label:
        preds[label.strip()] = prob
    res = {
        'PR-AUC': average_precision_score(y_test, prob),
        'F1':     f1_score(y_test, pred, zero_division=0),
        'AUC':    roc_auc_score(y_test, prob),
    }
    if label:
        print(f'  {label:<22} PR-AUC={res["PR-AUC"]:.4f}'
              f'  F1={res["F1"]:.4f}  AUC={res["AUC"]:.4f}')
    return res

def rare_attack_recall(prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        t: (pred[atk_test==t]==1).mean()
        for t in rare_types if (atk_test==t).sum() > 0
    }

def tail_recall(prob, bottom_pct=0.2, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    counts = {t:(atk_test==t).sum()
              for t in np.unique(atk_test) if t != 'Normal'}
    tail = sorted(counts, key=counts.get)[:max(1,int(len(counts)*bottom_pct))]
    recalls = [(pred[atk_test==t]==1).mean()
               for t in tail if (atk_test==t).sum()>0]
    return (np.mean(recalls) if recalls else 0.0), tail

print('평가 함수 준비 완료')
print(f'희귀 holdout: {rare_types}')

평가 함수 준비 완료
희귀 holdout: {'Worms', 'Shellcode'}


In [5]:
# ── 샘플러 정의 ──────────────────────────────────────────────
N_AG = 20; N_IT = 100; ALPHA = 0.2
N_TYPES = 9; BETA = 0.08; W_REPEL = 2.0; M_LOW = -0.5
N_TARGET = N_NORMAL

def _norm(X):
    mn, mx = X.min(0), X.max(0)
    r = np.where(mx-mn > 1e-8, mx-mn, 1.0)
    return (X - mn) / r

def build_train(X_norm_s, X_anom_s, idx):
    Xtr = np.vstack([X_norm_s, X_anom_s[idx]])
    ytr = np.array([0]*len(X_norm_s) + [1]*len(idx))
    return Xtr, ytr

X_norm = X_imbal[y_imbal==0]
X_anom = X_imbal[y_imbal==1]

# ── 룰 기반 ──
def run_random(X_anom, n, seed):
    return np.random.RandomState(seed).choice(len(X_anom), n, replace=True)

def run_kmeans(X_anom, n, seed, k=9):
    km = KMeans(k, random_state=seed, n_init=5).fit(X_anom)
    rng2 = np.random.RandomState(seed)
    idx = []
    for c in range(k):
        pool = np.where(km.labels_==c)[0]
        if len(pool): idx.extend(rng2.choice(pool, n//k, replace=True))
    while len(idx) < n: idx.append(rng2.randint(len(X_anom)))
    return np.array(idx[:n])

def run_greedy(X_anom, n, seed, k=5):
    nn = NearestNeighbors(n_neighbors=min(k+1,len(X_anom))).fit(X_anom)
    d, _ = nn.kneighbors(X_anom)
    density = 1.0 / (d[:,1:].mean(1) + 1e-8)
    probs = 1.0/(density+1e-8); probs /= probs.sum()
    return np.random.RandomState(seed).choice(len(X_anom), n, replace=True, p=probs)

def run_topdensity(X_anom, n, seed, k=5):
    nn = NearestNeighbors(n_neighbors=min(k+1,len(X_anom))).fit(X_anom)
    d, _ = nn.kneighbors(X_anom)
    density = 1.0 / (d[:,1:].mean(1) + 1e-8)
    probs = density / density.sum()
    return np.random.RandomState(seed).choice(len(X_anom), n, replace=True, p=probs)

# ── ACO ──
def run_aco(X_anom, n, seed):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a = len(Xn)
    ph = np.ones(N_a); visit = np.zeros(N_a)
    for _ in range(N_IT):
        for _ in range(N_AG):
            i = rng2.choice(N_a, p=ph/ph.sum())
            d = np.linalg.norm(Xn-Xn[i], axis=1); d[i]=1e9
            cands = np.argsort(d)[:10]
            j = cands[np.argmax(ph[cands])]
            nn = np.argmin(np.linalg.norm(Xn-np.clip((Xn[i]+Xn[j])/2,0,1), axis=1))
            visit[nn] += 1
        ph = ph*0.95 + visit*0.1
    probs = visit+1.0; probs /= probs.sum()
    return rng2.choice(N_a, n, replace=True, p=probs)

# ── PSO ──
def run_pso(X_anom, n, seed):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V = np.zeros_like(X); pX = X.copy()
    pS = np.array([-np.min(np.linalg.norm(Xn-X[i],axis=1)) for i in range(N_AG)])
    gi = np.argmax(pS); gX = X[gi].copy(); visit = np.zeros(N_a)
    for _ in range(N_IT):
        r1,r2 = rng2.rand(N_AG,D), rng2.rand(N_AG,D)
        V = 0.729*V + 1.494*r1*(pX-X) + 1.494*r2*(gX-X)
        X = np.clip(X+ALPHA*V, 0, 1)
        for i in range(N_AG):
            nn = np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc = -np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
        gi=np.argmax(pS); gX=pX[gi].copy()
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

# ── SPSO ──
def run_spso(X_anom, n, seed, r_s=0.3):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V = np.zeros_like(X); pX = X.copy()
    pS = -np.ones(N_AG)*1e9; visit = np.zeros(N_a)
    for _ in range(N_IT):
        sp_best = []
        for i in range(N_AG):
            sp = np.where(np.linalg.norm(X-X[i],axis=1)<=r_s)[0]
            sp_best.append(sp[np.argmax(pS[sp])])
        sbX = np.array([X[sp_best[i]] for i in range(N_AG)])
        r1,r2 = rng2.rand(N_AG,D), rng2.rand(N_AG,D)
        V = 0.729*V + 1.494*r1*(pX-X) + 1.494*r2*(sbX-X)
        X = np.clip(X+ALPHA*V, 0, 1)
        for i in range(N_AG):
            nn = np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc = -np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

# ── Crowding-DE ──
def run_cde(X_anom, n, seed, F=0.8, CR=0.9):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    pop = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    visit = np.zeros(N_a)
    for _ in range(N_IT):
        for i in range(N_AG):
            idxs = rng2.choice([j for j in range(N_AG) if j!=i], 3, replace=False)
            mutant = np.clip(pop[idxs[0]]+F*(pop[idxs[1]]-pop[idxs[2]]),0,1)
            trial  = np.where(rng2.rand(D)<CR, mutant, pop[i])
            most_sim = np.argmin(np.linalg.norm(pop-trial,axis=1))
            nn = np.argmin(np.linalg.norm(Xn-trial,axis=1))
            pop[most_sim]=Xn[nn]; visit[nn]+=1
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

# ── AISO ──
def run_aiso(X_anom, n, seed):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    W = rng2.dirichlet(np.ones(N_TYPES), N_AG)
    M = rng2.uniform(M_LOW, W_REPEL, (N_TYPES,N_TYPES))
    visit = np.zeros(N_a); w_r = W_REPEL
    for t in range(N_IT):
        if t%10==0:
            div = np.mean([np.linalg.norm(X[i]-X[j])
                           for i in range(N_AG) for j in range(i+1,N_AG)])
            w_r = 1.0 + 3.0*np.exp(-div/0.12)
        C = W@M@W.T; np.fill_diagonal(C,0)
        for i in range(N_AG):
            ci=C[i].copy(); ci[i]=0
            ta=np.argsort(ci)[-3:]; tr2=np.argsort(ci)[:3]
            Fv  = sum(ci[j]*(X[j]-X[i]) for j in ta)
            Fv += w_r*sum(ci[j]*(X[j]-X[i]) for j in tr2)
            Fv /= 6.0
            nn = np.argmin(np.linalg.norm(Xn-np.clip(X[i]+ALPHA*Fv,0,1),axis=1))
            X[i]=Xn[nn]; visit[nn]+=1.0
            bja=ta[np.argmax(ci[ta])]
            W[i]=(1-BETA)*W[i]+BETA*W[bja]; W[i]/=W[i].sum()
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

print('샘플러 11종 준비 완료')

샘플러 11종 준비 완료


In [6]:
# ── 전체 14개 메서드 실행 ────────────────────────────────────
def run(name, X_tr, y_tr):
    results[name] = evaluate(X_tr, y_tr, name)

print('='*65)
print('카테고리 1: 베이스라인')
run('원본(불균형)', X_imbal, y_imbal)

from sklearn.ensemble import GradientBoostingClassifier
clf_cw = GradientBoostingClassifier(n_estimators=150, random_state=SEED)
clf_cw.fit(X_imbal, y_imbal,
           sample_weight=np.where(y_imbal==1, N_NORMAL/N_SEEN, 1.0))
prob_cw = clf_cw.predict_proba(X_test)[:,1]
preds['Class Weight'] = prob_cw
results['Class Weight'] = {
    'PR-AUC': average_precision_score(y_test, prob_cw),
    'F1':     f1_score(y_test,(prob_cw>=0.5).astype(int),zero_division=0),
    'AUC':    roc_auc_score(y_test, prob_cw),
}
print(f'  {"Class Weight":<22} PR-AUC={results["Class Weight"]["PR-AUC"]:.4f}'
      f'  F1={results["Class Weight"]["F1"]:.4f}')

print('='*65)
print('카테고리 2: 룰 기반 오버샘플링')
print('  Random...', end=' ')
run('Random',      *build_train(X_norm, X_anom, run_random(X_anom, N_TARGET, SEED)))
print('  K-Means...', end=' ')
run('K-Means',     *build_train(X_norm, X_anom, run_kmeans(X_anom, N_TARGET, SEED)))
print('  Greedy...', end=' ')
run('Greedy',      *build_train(X_norm, X_anom, run_greedy(X_anom, N_TARGET, SEED)))
print('  Top-density...', end=' ')
run('Top-density', *build_train(X_norm, X_anom, run_topdensity(X_anom, N_TARGET, SEED)))

print('='*65)
print('카테고리 3: 합성 오버샘플링')
print('  RandomOver...', end=' ')
run('RandomOver', *RandomOverSampler(random_state=SEED).fit_resample(X_imbal, y_imbal))
print('  SMOTE...', end=' ')
run('SMOTE', *SMOTE(random_state=SEED, k_neighbors=5).fit_resample(X_imbal, y_imbal))
print('  ADASYN...', end=' ')
try:
    run('ADASYN', *ADASYN(random_state=SEED).fit_resample(X_imbal, y_imbal))
except Exception as e:
    results['ADASYN'] = results['SMOTE'].copy()
    preds['ADASYN']   = preds.get('SMOTE')
    print(f'fallback to SMOTE ({e})')

print('='*65)
print('카테고리 4: 최적화 샘플링 (PCA-20D 공간)')
print('  ACO...', end=' ')
run('ACO',  *build_train(X_norm, X_anom, run_aco(X_anom_pca,  N_TARGET, SEED)))
print('  PSO...', end=' ')
run('PSO',  *build_train(X_norm, X_anom, run_pso(X_anom_pca,  N_TARGET, SEED)))
print('  SPSO...', end=' ')
run('SPSO', *build_train(X_norm, X_anom, run_spso(X_anom_pca, N_TARGET, SEED)))
print('  CDE...', end=' ')
run('CDE',  *build_train(X_norm, X_anom, run_cde(X_anom_pca,  N_TARGET, SEED)))
print('  AISO...', end=' ')
run('AISO', *build_train(X_norm, X_anom, run_aiso(X_anom_pca, N_TARGET, SEED)))

print('='*65)
print(f'완료! 총 {len(results)}개 메서드')

카테고리 1: 베이스라인
  원본(불균형)                PR-AUC=0.9927  F1=0.8782  AUC=0.9845
  Class Weight           PR-AUC=0.9930  F1=0.9156
카테고리 2: 룰 기반 오버샘플링
  Random...   Random                 PR-AUC=0.9930  F1=0.9153  AUC=0.9853
  K-Means...   K-Means                PR-AUC=0.9918  F1=0.8988  AUC=0.9826
  Greedy...   Greedy                 PR-AUC=0.9921  F1=0.9128  AUC=0.9839
  Top-density...   Top-density            PR-AUC=0.9072  F1=0.5544  AUC=0.8022
카테고리 3: 합성 오버샘플링
  RandomOver...   RandomOver             PR-AUC=0.9928  F1=0.9144  AUC=0.9849
  SMOTE...   SMOTE                  PR-AUC=0.9920  F1=0.8983  AUC=0.9832
  ADASYN...   ADASYN                 PR-AUC=0.9914  F1=0.8962  AUC=0.9817
카테고리 4: 최적화 샘플링 (PCA-20D 공간)
  ACO...   ACO                    PR-AUC=0.9929  F1=0.9096  AUC=0.9849
  PSO...   PSO                    PR-AUC=0.9922  F1=0.9094  AUC=0.9829
  SPSO...   SPSO                   PR-AUC=0.9921  F1=0.9098  AUC=0.9828
  CDE...   CDE                    PR-AUC=0.9919  F1=0.9163  AUC=0.98

In [7]:
# ── 메인 결과 시각화 ─────────────────────────────────────────
CATEGORIES = {
    '베이스라인'  : ['원본(불균형)', 'Class Weight'],
    '룰 기반'     : ['Random', 'K-Means', 'Greedy', 'Top-density'],
    '합성 오버샘플': ['RandomOver', 'SMOTE', 'ADASYN'],
    '최적화 샘플링': ['ACO', 'PSO', 'SPSO', 'CDE', 'AISO'],
}
CAT_C = {
    '베이스라인'  : '#888888',
    '룰 기반'     : '#4C72B0',
    '합성 오버샘플': '#CCB974',
    '최적화 샘플링': '#C44E52',
}
MC = {m:CAT_C[c] for c,ms in CATEGORIES.items() for m in ms}
MC['AISO'] = '#8B0000'

ranking = sorted(results, key=lambda k: results[k]['PR-AUC'], reverse=True)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, metric in zip(axes[:2], ['PR-AUC','F1']):
    vals   = [results[m][metric] for m in ranking]
    colors = [MC.get(m,'#aaa') for m in ranking]
    bars   = ax.barh(ranking[::-1], vals[::-1], color=colors[::-1], alpha=0.85)
    for bar, v in zip(bars, vals[::-1]):
        ax.text(v+0.001, bar.get_y()+bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=8)
    ax.set_xlabel(metric)
    ax.set_title(f'UNSW-NB15 — {metric}\n(14 methods | 20K+2K train | Worms/Shellcode holdout)')
    ax.set_xlim(0, max(vals)*1.18)

ax = axes[2]
for m in ranking:
    ax.scatter(results[m]['AUC'], results[m]['PR-AUC'],
               s=160, color=MC.get(m,'#aaa'), zorder=5)
    ax.annotate(m, (results[m]['AUC'], results[m]['PR-AUC']),
                xytext=(4,4), textcoords='offset points', fontsize=8)
ax.set_xlabel('AUC-ROC'); ax.set_ylabel('PR-AUC')
ax.set_title('AUC vs PR-AUC')

from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(facecolor=CAT_C[c],label=c) for c in CAT_C],
               fontsize=8, loc='lower right')
plt.suptitle('UNSW-NB15 Showdown — 14 Methods\n'
             '(257K 레코드 | 9 attack types | Worms 44개/Shellcode 378개 train 제외)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('unsw_showdown_main.png', bbox_inches='tight', dpi=120)
plt.show()

print('\n' + '='*72)
print(f'  {"전략":<18} {"PR-AUC":>8} {"F1":>8} {"AUC":>8}  카테고리')
print('-'*72)
for rank, m in enumerate(ranking, 1):
    cat  = next((c for c,ms in CATEGORIES.items() if m in ms), '?')
    star = ' ★' if m=='AISO' else ''
    print(f'  {rank:>2}위 {m:<16} {results[m]["PR-AUC"]:>8.4f}'
          f' {results[m]["F1"]:>8.4f} {results[m]["AUC"]:>8.4f}  {cat}{star}')
print('='*72)


  전략                   PR-AUC       F1      AUC  카테고리
------------------------------------------------------------------------
   1위 Random             0.9930   0.9153   0.9853  룰 기반
   2위 Class Weight       0.9930   0.9156   0.9853  베이스라인
   3위 ACO                0.9929   0.9096   0.9849  최적화 샘플링
   4위 RandomOver         0.9928   0.9144   0.9849  합성 오버샘플
   5위 원본(불균형)            0.9927   0.8782   0.9845  베이스라인
   6위 AISO               0.9922   0.9067   0.9834  최적화 샘플링 ★
   7위 PSO                0.9922   0.9094   0.9829  최적화 샘플링
   8위 Greedy             0.9921   0.9128   0.9839  룰 기반
   9위 SPSO               0.9921   0.9098   0.9828  최적화 샘플링
  10위 SMOTE              0.9920   0.8983   0.9832  합성 오버샘플
  11위 CDE                0.9919   0.9163   0.9830  최적화 샘플링
  12위 K-Means            0.9918   0.8988   0.9826  룰 기반
  13위 ADASYN             0.9914   0.8962   0.9817  합성 오버샘플
  14위 Top-density        0.9072   0.5544   0.8022  룰 기반


In [8]:
# ── 희귀 공격 Recall 분석 ────────────────────────────────────
print('희귀 공격 Recall\n')
rare_results = {}
for m, prob in preds.items():
    if prob is None: continue
    rr = rare_attack_recall(prob)
    tr, _ = tail_recall(prob)
    rare_results[m] = {'rare': rr, 'tail_avg': tr}

all_rare = sorted(rare_types)
header = f'  {"방법":<18}' + ''.join(f'{t[:12]:>14}' for t in all_rare) + f'  {"Tail avg":>10}'
print(header); print('-'*len(header))
for m in ranking:
    if m not in rare_results: continue
    rr = rare_results[m]['rare']; ta = rare_results[m]['tail_avg']
    row = f'  {m:<18}' + ''.join(f'{rr.get(t,0):>14.3f}' for t in all_rare)
    print(row + f'  {ta:>10.3f}' + (' ★' if m=='AISO' else ''))

# 시각화
fig, axes = plt.subplots(1, len(all_rare)+1, figsize=(6*(len(all_rare)+1), 7))
for ax, t in zip(axes[:-1], all_rare):
    methods = [m for m in ranking if m in rare_results]
    vals    = [rare_results[m]['rare'].get(t,0) for m in methods]
    ax.barh(methods[::-1], vals[::-1],
            color=[MC.get(m,'#aaa') for m in methods[::-1]], alpha=0.85)
    for i,v in enumerate(vals[::-1]):
        ax.text(v+0.01, i, f'{v:.3f}', va='center', fontsize=8)
    n_test = (atk_test==t).sum()
    ax.set_title(f'{t}\n(test {n_test}개)', fontweight='bold')
    ax.set_xlabel('Recall'); ax.set_xlim(0,1.3)

methods = [m for m in ranking if m in rare_results]
vals    = [rare_results[m]['tail_avg'] for m in methods]
axes[-1].barh(methods[::-1], vals[::-1],
              color=[MC.get(m,'#aaa') for m in methods[::-1]], alpha=0.85)
for i,v in enumerate(vals[::-1]):
    axes[-1].text(v+0.005, i, f'{v:.3f}', va='center', fontsize=8)
axes[-1].set_title('Tail Recall 평균\n(하위 20% 빈도)', fontweight='bold')
axes[-1].set_xlabel('Recall'); axes[-1].set_xlim(0,1.3)

plt.suptitle('UNSW-NB15 희귀/꼬리 공격 탐지 Recall',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('unsw_rare_recall.png', bbox_inches='tight', dpi=120)
plt.show()

희귀 공격 Recall

  방법                     Shellcode         Worms    Tail avg
------------------------------------------------------------
  Random                     0.831         0.931       0.931
  Class Weight               0.837         0.923       0.923
  ACO                        0.828         0.877       0.877
  RandomOver                 0.825         0.931       0.931
  원본(불균형)                    0.349         0.685       0.685
  AISO                       0.797         0.877       0.877 ★
  PSO                        0.822         0.885       0.885
  Greedy                     0.806         0.923       0.923
  SPSO                       0.805         0.869       0.869
  SMOTE                      0.756         0.792       0.792
  CDE                        0.809         0.931       0.931
  K-Means                    0.662         0.846       0.846
  ADASYN                     0.813         0.854       0.854
  Top-density                0.002         0.000       0.000


In [9]:
# ── Mode Collapse 분석: PSO vs AISO ─────────────────────────
TRACK_EVERY = 10
track_iters = list(range(0, N_IT+1, TRACK_EVERY))

def entropy(counts):
    p = counts/(counts.sum()+1e-9); p=p[p>0]
    return -np.sum(p*np.log(p+1e-9))

def track_pso(Xn, seed):
    rng2=np.random.RandomState(seed)
    N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V=np.zeros_like(X); pX=X.copy()
    pS=-np.ones(N_AG)*1e9; gi=0; gX=X[0].copy(); visit=np.zeros(N_a)
    disp_hist=[]; vent_hist=[]
    for t in range(N_IT):
        if t%TRACK_EVERY==0:
            d=np.mean([np.linalg.norm(X[i]-X[j])
                       for i in range(N_AG) for j in range(i+1,N_AG)])
            disp_hist.append(d); vent_hist.append(entropy(visit))
        r1,r2=rng2.rand(N_AG,D),rng2.rand(N_AG,D)
        V=0.729*V+1.494*r1*(pX-X)+1.494*r2*(gX-X)
        X=np.clip(X+ALPHA*V,0,1)
        for i in range(N_AG):
            nn=np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc=-np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
        gi=np.argmax(pS); gX=pX[gi].copy()
    disp_hist.append(np.mean([np.linalg.norm(X[i]-X[j])
                               for i in range(N_AG) for j in range(i+1,N_AG)]))
    vent_hist.append(entropy(visit))
    return disp_hist, vent_hist

def track_aiso(Xn, seed):
    rng2=np.random.RandomState(seed)
    N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    W=rng2.dirichlet(np.ones(N_TYPES),N_AG)
    M=rng2.uniform(M_LOW,W_REPEL,(N_TYPES,N_TYPES))
    visit=np.zeros(N_a); w_r=W_REPEL
    disp_hist=[]; vent_hist=[]; went_hist=[]
    for t in range(N_IT):
        if t%TRACK_EVERY==0:
            d=np.mean([np.linalg.norm(X[i]-X[j])
                       for i in range(N_AG) for j in range(i+1,N_AG)])
            disp_hist.append(d); vent_hist.append(entropy(visit))
            went_hist.append(np.mean([-np.sum(W[i]*np.log(W[i]+1e-9))
                                       for i in range(N_AG)]))
        if t%10==0:
            dv=np.mean([np.linalg.norm(X[i]-X[j])
                        for i in range(N_AG) for j in range(i+1,N_AG)])
            w_r=1.0+3.0*np.exp(-dv/0.12)
        C=W@M@W.T; np.fill_diagonal(C,0)
        for i in range(N_AG):
            ci=C[i].copy(); ci[i]=0
            ta=np.argsort(ci)[-3:]; tr2=np.argsort(ci)[:3]
            Fv=sum(ci[j]*(X[j]-X[i]) for j in ta)\
              +w_r*sum(ci[j]*(X[j]-X[i]) for j in tr2); Fv/=6.0
            nn=np.argmin(np.linalg.norm(Xn-np.clip(X[i]+ALPHA*Fv,0,1),axis=1))
            X[i]=Xn[nn]; visit[nn]+=1.0
            bja=ta[np.argmax(ci[ta])]
            W[i]=(1-BETA)*W[i]+BETA*W[bja]; W[i]/=W[i].sum()
    disp_hist.append(np.mean([np.linalg.norm(X[i]-X[j])
                               for i in range(N_AG) for j in range(i+1,N_AG)]))
    vent_hist.append(entropy(visit))
    went_hist.append(np.mean([-np.sum(W[i]*np.log(W[i]+1e-9)) for i in range(N_AG)]))
    return disp_hist, vent_hist, went_hist

print('Mode Collapse 추적 중...')
Xn_track = _norm(X_anom_pca)
print('  PSO...', end=' ', flush=True)
pso_disp, pso_vent = track_pso(Xn_track, SEED); print('완료')
print('  AISO...', end=' ', flush=True)
aiso_disp, aiso_vent, aiso_went = track_aiso(Xn_track, SEED); print('완료')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(track_iters, pso_disp,  'b-o', ms=4, label='PSO', lw=2)
axes[0].plot(track_iters, aiso_disp, 'r-o', ms=4, label='AISO', lw=2)
axes[0].set_title('Agent Dispersion', fontweight='bold')
axes[0].set_xlabel('Iteration'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(track_iters, pso_vent,  'b-o', ms=4, label='PSO', lw=2)
axes[1].plot(track_iters, aiso_vent, 'r-o', ms=4, label='AISO', lw=2)
axes[1].set_title('Visit Count Entropy H(visit)', fontweight='bold')
axes[1].set_xlabel('Iteration'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(track_iters, aiso_went, 'r-o', ms=4, label='AISO W-entropy', lw=2)
axes[2].axhline(np.log(N_TYPES), color='gray', ls='--',
                label=f'Max H=log({N_TYPES})')
axes[2].set_title('AISO Type Entropy H(W)', fontweight='bold')
axes[2].set_xlabel('Iteration'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Mode Collapse 분석: PSO vs AISO (UNSW-NB15)',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('unsw_mode_collapse.png', bbox_inches='tight', dpi=120)
plt.show()

print(f'\n최종 dispersion : PSO={pso_disp[-1]:.4f}  AISO={aiso_disp[-1]:.4f}')
print(f'최종 visit H    : PSO={pso_vent[-1]:.4f}  AISO={aiso_vent[-1]:.4f}')
print(f'최종 W entropy  : AISO={aiso_went[-1]:.4f} (max={np.log(N_TYPES):.4f})')

Mode Collapse 추적 중...
  PSO... 완료
  AISO... 완료

최종 dispersion : PSO=0.1885  AISO=0.3594
최종 visit H    : PSO=4.3343  AISO=3.0691
최종 W entropy  : AISO=1.9108 (max=2.1972)


In [10]:
# ── Minority Coverage Entropy ────────────────────────────────
print('Coverage 계산 중...')
coverage = {}
atk_anom_train = atk_imbal_a.reset_index(drop=True).values

for name, fn, arg in [
    ('Random',     run_random,    (X_anom,     N_TARGET, SEED)),
    ('K-Means',    run_kmeans,    (X_anom,     N_TARGET, SEED)),
    ('Greedy',     run_greedy,    (X_anom,     N_TARGET, SEED)),
    ('Top-density',run_topdensity,(X_anom,     N_TARGET, SEED)),
    ('ACO',        run_aco,       (X_anom_pca, N_TARGET, SEED)),
    ('PSO',        run_pso,       (X_anom_pca, N_TARGET, SEED)),
    ('SPSO',       run_spso,      (X_anom_pca, N_TARGET, SEED)),
    ('CDE',        run_cde,       (X_anom_pca, N_TARGET, SEED)),
    ('AISO',       run_aiso,      (X_anom_pca, N_TARGET, SEED)),
]:
    idx = fn(*arg)
    vc  = pd.Series(atk_anom_train[idx]).value_counts()
    p   = vc.values/vc.values.sum()
    H   = -np.sum(p*np.log(p+1e-9))
    coverage[name] = {'entropy':H, 'n_types':len(vc), 'dist':vc}
    print(f'  {name:<18} H={H:.3f}  types={len(vc)}')

types_ro = atk_anom_train[
    np.random.RandomState(SEED).choice(len(atk_anom_train),N_TARGET,replace=True)]
vc_ro = pd.Series(types_ro).value_counts()
p_ro  = vc_ro.values/vc_ro.values.sum()
coverage['RandomOver'] = {
    'entropy': -np.sum(p_ro*np.log(p_ro+1e-9)),
    'n_types': len(vc_ro), 'dist': vc_ro
}

ordered = sorted(coverage, key=lambda k: coverage[k]['entropy'], reverse=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

entropies = [coverage[m]['entropy'] for m in ordered]
bars = axes[0].barh(ordered[::-1], entropies[::-1],
                    color=[MC.get(m,'#aaa') for m in ordered[::-1]], alpha=0.85)
for bar,v in zip(bars, entropies[::-1]):
    axes[0].text(v+0.02, bar.get_y()+bar.get_height()/2,
                 f'{v:.3f}', va='center', fontsize=9)
axes[0].axvline(np.log(len(atk_imbal_a.unique())), color='gray', ls='--',
                label=f'Max H=log({len(atk_imbal_a.unique())})')
axes[0].set_xlabel('공격 유형 분포 엔트로피 H')
axes[0].set_title('Minority Coverage Entropy\n(높을수록 다양한 공격 유형 포함)',
                  fontweight='bold')
axes[0].legend()

aiso_d = coverage.get('AISO',{}).get('dist', pd.Series())
pso_d  = coverage.get('PSO', {}).get('dist', pd.Series())
all_t  = sorted(set(list(aiso_d.index)+list(pso_d.index)))
x = np.arange(len(all_t)); w=0.35
aiso_p = np.array([aiso_d.get(t,0) for t in all_t])/(sum(aiso_d)+1e-9)
pso_p  = np.array([pso_d.get(t,0)  for t in all_t])/(sum(pso_d)+1e-9)
axes[1].bar(x-w/2, aiso_p, w, label='AISO', color='#8B0000', alpha=0.8)
axes[1].bar(x+w/2, pso_p,  w, label='PSO',  color='blue',    alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(all_t, rotation=45, ha='right', fontsize=8)
axes[1].set_title('AISO vs PSO: 오버샘플 공격 유형 분포',
                  fontweight='bold')
axes[1].set_ylabel('비율'); axes[1].legend()

plt.suptitle('UNSW-NB15 Minority Coverage Entropy',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('unsw_minority_coverage.png', bbox_inches='tight', dpi=120)
plt.show()

Coverage 계산 중...
  Random             H=1.509  types=7
  K-Means            H=1.327  types=7
  Greedy             H=1.453  types=7
  Top-density        H=0.687  types=5
  ACO                H=1.543  types=7
  PSO                H=1.615  types=7
  SPSO               H=1.574  types=7
  CDE                H=1.504  types=7
  AISO               H=1.548  types=7


In [11]:
# ── N_TYPES 차원 탐색: AISO 최적 타입 수 찾기 ────────────────────────────────
TEST_TYPES = [4, 6, 8, 9, 12, 14, 17, 20, 24]

def run_aiso_nt(X_anom, n, seed, n_types):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a, N_AG, replace=True)].copy().astype(float)
    W = rng2.dirichlet(np.ones(n_types), N_AG)
    M = rng2.uniform(M_LOW, W_REPEL, (n_types, n_types))
    visit = np.zeros(N_a); w_r = W_REPEL
    for t in range(N_IT):
        if t % 10 == 0:
            div = np.mean([np.linalg.norm(X[i]-X[j])
                           for i in range(N_AG) for j in range(i+1, N_AG)])
            w_r = 1.0 + 3.0 * np.exp(-div / 0.12)
        C = W @ M @ W.T; np.fill_diagonal(C, 0)
        for i in range(N_AG):
            ci = C[i].copy(); ci[i] = 0
            ta = np.argsort(ci)[-3:]; tr2 = np.argsort(ci)[:3]
            Fv  = sum(ci[j] * (X[j]-X[i]) for j in ta)
            Fv += w_r * sum(ci[j] * (X[j]-X[i]) for j in tr2)
            Fv /= 6.0
            nn = np.argmin(np.linalg.norm(Xn - np.clip(X[i]+ALPHA*Fv, 0, 1), axis=1))
            X[i] = Xn[nn]; visit[nn] += 1.0
            bja = ta[np.argmax(ci[ta])]
            W[i] = (1-BETA)*W[i] + BETA*W[bja]; W[i] /= W[i].sum()
    probs = visit + 1.0; probs /= probs.sum()
    return rng2.choice(N_a, n, replace=True, p=probs)

print('N_TYPES 차원 탐색 (UNSW-NB15)')
print(f'  현재 설정: N_TYPES={N_TYPES}')
print('-'*45)
type_results_nt = {}
for nt in TEST_TYPES:
    idx = run_aiso_nt(X_anom_pca, N_TARGET, SEED, nt)
    Xtr, ytr = build_train(X_norm, X_anom, idx)
    res = evaluate(Xtr, ytr)
    type_results_nt[nt] = res['PR-AUC']
    marker = ' <-- 현재' if nt == N_TYPES else ''
    print(f'  N_TYPES={nt:>2}  PR-AUC={res["PR-AUC"]:.4f}{marker}')

best_nt = max(type_results_nt, key=type_results_nt.get)
print('-'*45)
print(f'  최적 N_TYPES = {best_nt}  (PR-AUC={type_results_nt[best_nt]:.4f})')
print(f'  기본값({N_TYPES}) PR-AUC  = {type_results_nt[N_TYPES]:.4f}')
print(f'  개선 여지        = {type_results_nt[best_nt]-type_results_nt[N_TYPES]:+.4f}')


N_TYPES 차원 탐색 (UNSW-NB15)
  현재 설정: N_TYPES=9
---------------------------------------------
  N_TYPES= 4  PR-AUC=0.9924
  N_TYPES= 6  PR-AUC=0.9918
  N_TYPES= 8  PR-AUC=0.9924
  N_TYPES= 9  PR-AUC=0.9922 <-- 현재
  N_TYPES=12  PR-AUC=0.9920
  N_TYPES=14  PR-AUC=0.9924
  N_TYPES=17  PR-AUC=0.9923
  N_TYPES=20  PR-AUC=0.9922
  N_TYPES=24  PR-AUC=0.9922
---------------------------------------------
  최적 N_TYPES = 14  (PR-AUC=0.9924)
  기본값(9) PR-AUC  = 0.9922
  개선 여지        = +0.0002
